This notebook tests whether current Kalshi market features can predict the
price change over the following five minutes.

The notebook:

1. Loads eligible observations from the existing Gold feature table.
2. Keeps every row and both paired markets from one ESPN game in one split.
3. Creates reproducible train, validation, and test game groups.
4. Establishes a DummyRegressor baseline.
5. Trains Ridge regression and histogram gradient boosting.
6. Evaluates general prediction quality on validation games.
7. Measures the tradeoff between opportunity quality and opportunity frequency.
8. Collapses adjacent qualifying observations into approximate signal episodes.

Prediction percentiles are opportunity-selection thresholds, not anomaly scores.

The final test split must remain untouched until the model and opportunity
threshold have been selected using training and validation data.

In [0]:
# Add the repository root to the Python path
import sys
from pathlib import Path


def find_repository_root(start_path: Path) -> Path:
    """
    Walk upward until a directory containing src/modeling is found.
    """
    current_path = start_path.resolve()

    for candidate in [current_path, *current_path.parents]:
        if (candidate / "src" / "modeling").is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not find the repository root containing src/modeling. "
        f"Notebook working directory: {start_path}"
    )


REPOSITORY_ROOT = find_repository_root(Path.cwd())

if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

print(f"Repository root: {REPOSITORY_ROOT}")

In [0]:
# Imports and configuration
import copy

import numpy as np
import pandas as pd

from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from src.modeling.model_config import (
    GOLD_TABLE,
    IDENTIFIER_COLUMNS,
    FEATURE_COLUMNS,
    EVALUATION_COLUMNS,
    ELIGIBILITY_COLUMN,
    TARGET_COLUMN,
    GROUP_COLUMN,
    TRAIN_FRACTION,
    VALIDATION_FRACTION,
    TEST_FRACTION,
    RANDOM_STATE,
    RIDGE_PARAMETERS,
    HIST_GRADIENT_BOOSTING_PARAMETERS,
    OPPORTUNITY_PERCENTILES,
    SIGNAL_EPISODE_GAP_MINUTES,
)


split_fraction_sum = (
    TRAIN_FRACTION
    + VALIDATION_FRACTION
    + TEST_FRACTION
)

if not np.isclose(split_fraction_sum, 1.0):
    raise ValueError(
        "TRAIN_FRACTION, VALIDATION_FRACTION, and TEST_FRACTION "
        f"must sum to 1.0. Current sum: {split_fraction_sum}"
    )

print(f"Gold table: {GOLD_TABLE}")
print(f"Target: {TARGET_COLUMN}")
print(f"Game grouping column: {GROUP_COLUMN}")
print(f"Feature count: {len(FEATURE_COLUMNS)}")
print(f"Opportunity percentiles: {OPPORTUNITY_PERCENTILES}")

In [0]:
# Load eligible Gold observations
from pyspark.sql import functions as F


CONTEXT_COLUMNS = [
    "is_trade_eligible",
    "selection_team",
    "opponent_team",
    "market_side",
    "game_start_utc",
]

required_columns = list(
    dict.fromkeys(
        IDENTIFIER_COLUMNS
        + FEATURE_COLUMNS
        + EVALUATION_COLUMNS
        + [ELIGIBILITY_COLUMN]
        + CONTEXT_COLUMNS
    )
)

gold_df = (
    spark.read.table(GOLD_TABLE)
    .select(*required_columns)
    .filter(F.col(ELIGIBILITY_COLUMN) == F.lit(True))
    .dropna(
        subset=(
            FEATURE_COLUMNS
            + [
                TARGET_COLUMN,
                GROUP_COLUMN,
                "kalshi_ticker",
                "minute_ts",
            ]
        )
    )
)

print(f"Loaded table: {GOLD_TABLE}")
print(f"Selected columns: {len(gold_df.columns)}")

display(gold_df.limit(20))

In [0]:
# Profile the eligible modeling dataset
dataset_profile_df = gold_df.agg(
    F.count("*").alias("row_count"),
    F.countDistinct("kalshi_ticker").alias("ticker_count"),
    F.countDistinct(GROUP_COLUMN).alias("game_count"),
    F.min("minute_ts").alias("minimum_minute_ts"),
    F.max("minute_ts").alias("maximum_minute_ts"),
)

display(dataset_profile_df)

In [0]:
# Check the paired-market structure
game_structure_df = (
    gold_df
    .groupBy(GROUP_COLUMN)
    .agg(
        F.countDistinct("kalshi_ticker").alias("ticker_count"),
        F.count("*").alias("observation_count"),
        F.min("game_start_utc").alias("game_start_utc"),
        F.collect_set("market_side").alias("market_sides"),
    )
)

display(
    game_structure_df
    .groupBy("ticker_count")
    .agg(
        F.count("*").alias("game_count")
    )
    .orderBy("ticker_count")
)

display(
    game_structure_df
    .orderBy(F.desc("ticker_count"))
    .limit(20)
)

In [0]:
# Convert to pandas
modeling_pdf = (
    gold_df
    .orderBy(
        F.col(GROUP_COLUMN),
        F.col("kalshi_ticker"),
        F.col("minute_ts"),
    )
    .toPandas()
)

modeling_pdf["minute_ts"] = pd.to_datetime(
    modeling_pdf["minute_ts"],
    utc=True,
)

modeling_pdf["game_start_utc"] = pd.to_datetime(
    modeling_pdf["game_start_utc"],
    utc=True,
)

print(f"Modeling observations: {len(modeling_pdf):,}")
print(
    f"Independent games: "
    f"{modeling_pdf[GROUP_COLUMN].nunique():,}"
)
print(
    f"Kalshi tickers: "
    f"{modeling_pdf['kalshi_ticker'].nunique():,}"
)

modeling_pdf.head()

In [0]:
# Define the grouped split function
def grouped_train_validation_test_split(
    dataframe: pd.DataFrame,
    group_column: str,
    train_fraction: float,
    validation_fraction: float,
    test_fraction: float,
    random_state: int,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Split a DataFrame while keeping every group entirely within one split.

    The first split removes the full test fraction.

    The second split divides the remaining development games into train and
    validation using a relative validation fraction.
    """
    fraction_sum = (
        train_fraction
        + validation_fraction
        + test_fraction
    )

    if not np.isclose(fraction_sum, 1.0):
        raise ValueError(
            f"Split fractions must sum to 1.0, not {fraction_sum}."
        )

    outer_splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=test_fraction,
        random_state=random_state,
    )

    development_indices, test_indices = next(
        outer_splitter.split(
            X=dataframe,
            groups=dataframe[group_column],
        )
    )

    development_df = dataframe.iloc[
        development_indices
    ].copy()

    test_df = dataframe.iloc[
        test_indices
    ].copy()

    development_fraction = (
        train_fraction
        + validation_fraction
    )

    relative_validation_fraction = (
        validation_fraction
        / development_fraction
    )

    inner_splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=relative_validation_fraction,
        random_state=random_state,
    )

    train_indices, validation_indices = next(
        inner_splitter.split(
            X=development_df,
            groups=development_df[group_column],
        )
    )

    train_df = development_df.iloc[
        train_indices
    ].copy()

    validation_df = development_df.iloc[
        validation_indices
    ].copy()

    return train_df, validation_df, test_df

In [0]:
# Create the train, validation, and test game groups
train_pdf, validation_pdf, test_pdf = (
    grouped_train_validation_test_split(
        dataframe=modeling_pdf,
        group_column=GROUP_COLUMN,
        train_fraction=TRAIN_FRACTION,
        validation_fraction=VALIDATION_FRACTION,
        test_fraction=TEST_FRACTION,
        random_state=RANDOM_STATE,
    )
)

split_summary_pdf = pd.DataFrame(
    [
        {
            "split": "train",
            "row_count": len(train_pdf),
            "game_count": train_pdf[
                GROUP_COLUMN
            ].nunique(),
            "ticker_count": train_pdf[
                "kalshi_ticker"
            ].nunique(),
        },
        {
            "split": "validation",
            "row_count": len(validation_pdf),
            "game_count": validation_pdf[
                GROUP_COLUMN
            ].nunique(),
            "ticker_count": validation_pdf[
                "kalshi_ticker"
            ].nunique(),
        },
        {
            "split": "test",
            "row_count": len(test_pdf),
            "game_count": test_pdf[
                GROUP_COLUMN
            ].nunique(),
            "ticker_count": test_pdf[
                "kalshi_ticker"
            ].nunique(),
        },
    ]
)

split_summary_pdf["row_fraction"] = (
    split_summary_pdf["row_count"]
    / len(modeling_pdf)
)

split_summary_pdf["game_fraction"] = (
    split_summary_pdf["game_count"]
    / modeling_pdf[GROUP_COLUMN].nunique()
)

display(split_summary_pdf)

In [0]:
# Prove that no game crosses dataset splits
train_games = set(
    train_pdf[GROUP_COLUMN].unique()
)

validation_games = set(
    validation_pdf[GROUP_COLUMN].unique()
)

test_games = set(
    test_pdf[GROUP_COLUMN].unique()
)

assert train_games.isdisjoint(validation_games)
assert train_games.isdisjoint(test_games)
assert validation_games.isdisjoint(test_games)

assigned_games = (
    train_games
    | validation_games
    | test_games
)

all_games = set(
    modeling_pdf[GROUP_COLUMN].unique()
)

assert assigned_games == all_games

print("Game-level leakage checks passed.")
print("Every ESPN game belongs to exactly one split.")
print("Both paired markets and all minutes remain together.")

In [0]:
# Prepare feature and target datasets
X_train = train_pdf[FEATURE_COLUMNS]
y_train = train_pdf[TARGET_COLUMN]

X_validation = validation_pdf[FEATURE_COLUMNS]
y_validation = validation_pdf[TARGET_COLUMN]

X_test = test_pdf[FEATURE_COLUMNS]
y_test = test_pdf[TARGET_COLUMN]

print(f"Training feature shape: {X_train.shape}")
print(f"Validation feature shape: {X_validation.shape}")
print(f"Test feature shape: {X_test.shape}")

print()
print("Training target distribution:")
display(y_train.describe(percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]))

In [0]:
# Define the baseline and candidate models
def build_candidate_models() -> dict[str, Pipeline]:
    """
    Build fresh model pipelines.

    DummyRegressor is a baseline only. It predicts the training-set mean
    and establishes the minimum performance a useful model should beat.
    """
    return {
        "dummy_mean": Pipeline(
            steps=[
                (
                    "imputer",
                    SimpleImputer(strategy="median"),
                ),
                (
                    "model",
                    DummyRegressor(strategy="mean"),
                ),
            ]
        ),

        "ridge": Pipeline(
            steps=[
                (
                    "imputer",
                    SimpleImputer(strategy="median"),
                ),
                (
                    "scaler",
                    StandardScaler(),
                ),
                (
                    "model",
                    Ridge(**RIDGE_PARAMETERS),
                ),
            ]
        ),

        "hist_gradient_boosting": Pipeline(
            steps=[
                (
                    "imputer",
                    SimpleImputer(strategy="median"),
                ),
                (
                    "model",
                    HistGradientBoostingRegressor(
                        **HIST_GRADIENT_BOOSTING_PARAMETERS
                    ),
                ),
            ]
        ),
    }


candidate_models = build_candidate_models()

print(f"Candidate models: {list(candidate_models)}")

In [0]:
# Define general regression metrics
def calculate_regression_metrics(
    actual: pd.Series | np.ndarray,
    predicted: np.ndarray,
) -> dict[str, float]:
    """
    Calculate general regression metrics.

    Directional accuracy is included as a secondary metric because a large
    portion of five-minute price changes may equal zero.
    """
    actual_array = np.asarray(actual)
    predicted_array = np.asarray(predicted)

    nonzero_actual_mask = actual_array != 0

    if nonzero_actual_mask.any():
        directional_accuracy_nonzero = np.mean(
            np.sign(predicted_array[nonzero_actual_mask])
            == np.sign(actual_array[nonzero_actual_mask])
        )
    else:
        directional_accuracy_nonzero = np.nan

    return {
        "mae": mean_absolute_error(
            actual_array,
            predicted_array,
        ),
        "rmse": np.sqrt(
            mean_squared_error(
                actual_array,
                predicted_array,
            )
        ),
        "r2": r2_score(
            actual_array,
            predicted_array,
        ),
        "prediction_mean": predicted_array.mean(),
        "prediction_stddev": predicted_array.std(),
        "directional_accuracy_all": np.mean(
            np.sign(predicted_array)
            == np.sign(actual_array)
        ),
        "directional_accuracy_nonzero": (
            directional_accuracy_nonzero
        ),
    }

In [0]:
# Train the candidate models
fitted_models = {}
validation_predictions = {}
validation_metric_records = []

for model_name, model in candidate_models.items():
    fitted_model = copy.deepcopy(model)

    fitted_model.fit(
        X_train,
        y_train,
    )

    predictions = fitted_model.predict(
        X_validation
    )

    fitted_models[model_name] = fitted_model
    validation_predictions[model_name] = predictions

    metrics = calculate_regression_metrics(
        actual=y_validation,
        predicted=predictions,
    )

    validation_metric_records.append(
        {
            "model_name": model_name,
            **metrics,
        }
    )

validation_metrics_pdf = (
    pd.DataFrame(validation_metric_records)
    .sort_values(
        by="mae",
        ascending=True,
    )
    .reset_index(drop=True)
)

display(validation_metrics_pdf)

In [0]:
# Create a validation scoring DataFrame
def create_scored_dataframe(
    source_df: pd.DataFrame,
    predictions: np.ndarray,
) -> pd.DataFrame:
    """
    Attach model predictions to the source observations.
    """
    scored_df = source_df[
        IDENTIFIER_COLUMNS
        + [
            "current_price",
            "is_trade_eligible",
            "selection_team",
            "opponent_team",
            "market_side",
            "game_start_utc",
            TARGET_COLUMN,
        ]
    ].copy()

    scored_df["prediction"] = predictions

    return scored_df


validation_scored_by_model = {
    model_name: create_scored_dataframe(
        source_df=validation_pdf,
        predictions=predictions,
    )
    for model_name, predictions
    in validation_predictions.items()
}

In [0]:
# Learn opportunity thresholds from validation predictions
def calculate_opportunity_thresholds(
    predictions: pd.Series | np.ndarray,
    percentiles: list[float],
) -> dict[float, float]:
    """
    Calculate prediction thresholds from validation predictions.

    These thresholds must later be applied unchanged to the test set.
    They must not be recalculated from test predictions.
    """
    prediction_series = pd.Series(predictions)

    return {
        percentile: float(
            prediction_series.quantile(percentile)
        )
        for percentile in percentiles
    }


validation_thresholds_by_model = {
    model_name: calculate_opportunity_thresholds(
        predictions=scored_df["prediction"],
        percentiles=OPPORTUNITY_PERCENTILES,
    )
    for model_name, scored_df
    in validation_scored_by_model.items()
}

validation_thresholds_by_model

In [0]:
# Define signal-episode assignment
def assign_signal_episodes(
    signals_df: pd.DataFrame,
    episode_gap_minutes: int,
) -> pd.DataFrame:
    """
    Collapse adjacent qualifying observations into approximate signal episodes.

    Signals are grouped independently by game and Kalshi ticker.

    A new episode begins when the current qualifying observation is more than
    episode_gap_minutes after the previous qualifying observation.
    """
    if signals_df.empty:
        empty_df = signals_df.copy()
        empty_df["signal_episode_number"] = pd.Series(dtype="int64")
        empty_df["signal_episode_id"] = pd.Series(dtype="string")
        return empty_df

    episode_df = signals_df.sort_values(
        by=[
            GROUP_COLUMN,
            "kalshi_ticker",
            "minute_ts",
        ]
    ).copy()

    prior_signal_ts = (
        episode_df
        .groupby(
            [
                GROUP_COLUMN,
                "kalshi_ticker",
            ]
        )["minute_ts"]
        .shift(1)
    )

    minutes_since_prior_signal = (
        episode_df["minute_ts"]
        - prior_signal_ts
    ).dt.total_seconds() / 60.0

    episode_df["is_new_signal_episode"] = (
        prior_signal_ts.isna()
        | (
            minutes_since_prior_signal
            > episode_gap_minutes
        )
    )

    episode_df["signal_episode_number"] = (
        episode_df
        .groupby(
            [
                GROUP_COLUMN,
                "kalshi_ticker",
            ]
        )["is_new_signal_episode"]
        .cumsum()
        .astype("int64")
    )

    episode_df["signal_episode_id"] = (
        episode_df[GROUP_COLUMN].astype(str)
        + "__"
        + episode_df["kalshi_ticker"].astype(str)
        + "__"
        + episode_df[
            "signal_episode_number"
        ].astype(str)
    )

    return episode_df

In [0]:
# Define opportunity quality and frequency evaluation
def evaluate_opportunity_thresholds(
    scored_df: pd.DataFrame,
    thresholds: dict[float, float],
    target_column: str,
    episode_gap_minutes: int,
) -> pd.DataFrame:
    """
    Evaluate both return quality and opportunity frequency.

    Raw qualifying rows are reported, but distinct signal episodes are the
    more realistic approximation of separate buying opportunities.
    """
    total_games = scored_df[
        GROUP_COLUMN
    ].nunique()

    total_tickers = scored_df[
        "kalshi_ticker"
    ].nunique()

    records = []

    for percentile, threshold in thresholds.items():
        signals_df = scored_df.loc[
            scored_df["prediction"] >= threshold
        ].copy()

        if signals_df.empty:
            records.append(
                {
                    "prediction_percentile": percentile,
                    "prediction_threshold": threshold,
                    "raw_signal_count": 0,
                    "signal_episode_count": 0,
                    "total_game_count": total_games,
                    "active_game_count": 0,
                    "game_coverage_rate": 0.0,
                    "raw_signals_per_game": 0.0,
                    "episodes_per_game": 0.0,
                    "mean_episodes_per_active_game": np.nan,
                    "median_episodes_per_active_game": np.nan,
                    "total_ticker_count": total_tickers,
                    "active_ticker_count": 0,
                    "mean_actual_price_change": np.nan,
                    "median_actual_price_change": np.nan,
                    "positive_return_rate": np.nan,
                    "nonnegative_return_rate": np.nan,
                    "mean_prediction": np.nan,
                }
            )
            continue

        episode_rows_df = assign_signal_episodes(
            signals_df=signals_df,
            episode_gap_minutes=episode_gap_minutes,
        )

        # Use the first qualifying observation as the representative entry
        # point for each signal episode.
        episode_entries_df = (
            episode_rows_df
            .sort_values("minute_ts")
            .drop_duplicates(
                subset=["signal_episode_id"],
                keep="first",
            )
        )

        episodes_per_game = (
            episode_entries_df
            .groupby(GROUP_COLUMN)
            .size()
        )

        active_game_count = (
            episode_entries_df[
                GROUP_COLUMN
            ].nunique()
        )

        active_ticker_count = (
            episode_entries_df[
                "kalshi_ticker"
            ].nunique()
        )

        actual_returns = episode_entries_df[
            target_column
        ]

        records.append(
            {
                "prediction_percentile": percentile,
                "prediction_threshold": threshold,
                "raw_signal_count": len(signals_df),
                "signal_episode_count": len(
                    episode_entries_df
                ),
                "total_game_count": total_games,
                "active_game_count": active_game_count,
                "game_coverage_rate": (
                    active_game_count
                    / total_games
                    if total_games > 0
                    else np.nan
                ),
                "raw_signals_per_game": (
                    len(signals_df)
                    / total_games
                    if total_games > 0
                    else np.nan
                ),
                "episodes_per_game": (
                    len(episode_entries_df)
                    / total_games
                    if total_games > 0
                    else np.nan
                ),
                "mean_episodes_per_active_game": (
                    episodes_per_game.mean()
                ),
                "median_episodes_per_active_game": (
                    episodes_per_game.median()
                ),
                "total_ticker_count": total_tickers,
                "active_ticker_count": (
                    active_ticker_count
                ),
                "mean_actual_price_change": (
                    actual_returns.mean()
                ),
                "median_actual_price_change": (
                    actual_returns.median()
                ),
                "positive_return_rate": (
                    actual_returns.gt(0).mean()
                ),
                "nonnegative_return_rate": (
                    actual_returns.ge(0).mean()
                ),
                "mean_prediction": (
                    episode_entries_df[
                        "prediction"
                    ].mean()
                ),
            }
        )

    return (
        pd.DataFrame(records)
        .sort_values("prediction_percentile")
        .reset_index(drop=True)
    )

In [0]:
# Evaluate validation opportunity curves
validation_opportunity_frames = []

for model_name, scored_df in validation_scored_by_model.items():
    model_opportunity_metrics = (
        evaluate_opportunity_thresholds(
            scored_df=scored_df,
            thresholds=(
                validation_thresholds_by_model[
                    model_name
                ]
            ),
            target_column=TARGET_COLUMN,
            episode_gap_minutes=(
                SIGNAL_EPISODE_GAP_MINUTES
            ),
        )
    )

    model_opportunity_metrics.insert(
        0,
        "model_name",
        model_name,
    )

    validation_opportunity_frames.append(
        model_opportunity_metrics
    )

validation_opportunity_metrics_pdf = pd.concat(
    validation_opportunity_frames,
    ignore_index=True,
)

display(
    validation_opportunity_metrics_pdf.sort_values(
        by=[
            "model_name",
            "prediction_percentile",
        ]
    )
)

In [0]:
# Compare opportunity frequency and quality directly
validation_opportunity_comparison_pdf = (
    validation_opportunity_metrics_pdf[
        [
            "model_name",
            "prediction_percentile",
            "prediction_threshold",
            "signal_episode_count",
            "active_game_count",
            "game_coverage_rate",
            "episodes_per_game",
            "mean_episodes_per_active_game",
            "mean_actual_price_change",
            "median_actual_price_change",
            "positive_return_rate",
        ]
    ]
    .sort_values(
        by=[
            "model_name",
            "prediction_percentile",
        ]
    )
    .reset_index(drop=True)
)

display(validation_opportunity_comparison_pdf)

In [0]:
# Calculate the unconditional validation baseline
validation_unconditional_baseline_pdf = pd.DataFrame(
    [
        {
            "validation_row_count": len(
                validation_pdf
            ),
            "validation_game_count": (
                validation_pdf[
                    GROUP_COLUMN
                ].nunique()
            ),
            "mean_future_price_change_5m": (
                validation_pdf[
                    TARGET_COLUMN
                ].mean()
            ),
            "median_future_price_change_5m": (
                validation_pdf[
                    TARGET_COLUMN
                ].median()
            ),
            "positive_return_rate": (
                validation_pdf[
                    TARGET_COLUMN
                ].gt(0).mean()
            ),
            "nonnegative_return_rate": (
                validation_pdf[
                    TARGET_COLUMN
                ].ge(0).mean()
            ),
        }
    ]
)

display(validation_unconditional_baseline_pdf)

In [0]:
# Inspect Ridge coefficients
ridge_pipeline = fitted_models["ridge"]
ridge_model = ridge_pipeline.named_steps["model"]

ridge_coefficients_pdf = (
    pd.DataFrame(
        {
            "feature": FEATURE_COLUMNS,
            "coefficient": ridge_model.coef_,
        }
    )
    .assign(
        absolute_coefficient=lambda df: (
            df["coefficient"].abs()
        )
    )
    .sort_values(
        by="absolute_coefficient",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(ridge_coefficients_pdf)

In [0]:
# Inspect gradient-boosting permutation importance
hist_gradient_boosting_model = fitted_models[
    "hist_gradient_boosting"
]

permutation_results = permutation_importance(
    estimator=hist_gradient_boosting_model,
    X=X_validation,
    y=y_validation,
    scoring="neg_mean_absolute_error",
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

hist_gradient_boosting_importance_pdf = (
    pd.DataFrame(
        {
            "feature": FEATURE_COLUMNS,
            "importance_mean": (
                permutation_results.importances_mean
            ),
            "importance_stddev": (
                permutation_results.importances_std
            ),
        }
    )
    .sort_values(
        by="importance_mean",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(hist_gradient_boosting_importance_pdf)

In [0]:
# Select the candidate model using validation data
non_dummy_validation_metrics_pdf = (
    validation_metrics_pdf.loc[
        validation_metrics_pdf["model_name"]
        != "dummy_mean"
    ]
    .copy()
)

selected_model_name = (
    non_dummy_validation_metrics_pdf
    .sort_values(
        by="mae",
        ascending=True,
    )
    .iloc[0]["model_name"]
)

selected_validation_model = fitted_models[
    selected_model_name
]

selected_validation_thresholds = (
    validation_thresholds_by_model[
        selected_model_name
    ]
)

print(
    "Candidate selected by validation MAE: "
    f"{selected_model_name}"
)

display(
    validation_opportunity_metrics_pdf.loc[
        validation_opportunity_metrics_pdf[
            "model_name"
        ]
        == selected_model_name
    ]
)

## Final Test-Set Gate

Do not run the remaining cells until:

1. The candidate model has been selected using validation data.
2. The opportunity percentile or practical percentile range has been chosen.
3. The signal-episode gap has been accepted as a reasonable diagnostic.
4. No further model or threshold changes will be made based on test results.

The test set is for one final unbiased evaluation.

In [0]:
# Set the chosen opportunity percentiles
#
# These percentiles were selected using validation results and practical
# strategy considerations. The corresponding numeric prediction thresholds
# must come from validation predictions, not from test predictions.

SELECTED_OPPORTUNITY_PERCENTILES = [
    0.95,
    0.975,
    0.99,
]

missing_percentiles = [
    percentile
    for percentile in SELECTED_OPPORTUNITY_PERCENTILES
    if percentile not in OPPORTUNITY_PERCENTILES
]

if missing_percentiles:
    raise ValueError(
        "Every selected opportunity percentile must appear in "
        f"OPPORTUNITY_PERCENTILES. Missing: {missing_percentiles}. "
        f"Configured values: {OPPORTUNITY_PERCENTILES}"
    )

selected_prediction_thresholds = {
    percentile: selected_validation_thresholds[percentile]
    for percentile in SELECTED_OPPORTUNITY_PERCENTILES
}

selected_thresholds_pdf = pd.DataFrame(
    [
        {
            "model_name": selected_model_name,
            "prediction_percentile": percentile,
            "validation_derived_threshold": threshold,
        }
        for percentile, threshold
        in selected_prediction_thresholds.items()
    ]
).sort_values(
    by="prediction_percentile"
).reset_index(drop=True)

print(
    "The following validation-derived thresholds will be "
    "applied unchanged to the test set:"
)

display(selected_thresholds_pdf)

In [0]:
# Refit the candidate model on train plus validation
development_pdf = pd.concat(
    [
        train_pdf,
        validation_pdf,
    ],
    ignore_index=True,
)

X_development = development_pdf[
    FEATURE_COLUMNS
]

y_development = development_pdf[
    TARGET_COLUMN
]

fresh_models = build_candidate_models()

final_candidate_model = fresh_models[
    selected_model_name
]

final_candidate_model.fit(
    X_development,
    y_development,
)

print(
    f"Refitted {selected_model_name} on "
    f"{len(development_pdf):,} observations from "
    f"{development_pdf[GROUP_COLUMN].nunique():,} games."
)

In [0]:
# Score the untouched test games
test_predictions = final_candidate_model.predict(
    X_test
)

test_scored_pdf = create_scored_dataframe(
    source_df=test_pdf,
    predictions=test_predictions,
)

test_regression_metrics = (
    calculate_regression_metrics(
        actual=y_test,
        predicted=test_predictions,
    )
)

test_regression_metrics_pdf = pd.DataFrame(
    [
        {
            "model_name": selected_model_name,
            **test_regression_metrics,
        }
    ]
)

display(test_regression_metrics_pdf)

In [0]:
# Apply all validation-derived thresholds to the untouched test data

test_signals_by_percentile = {}
test_episode_rows_by_percentile = {}
test_episode_entries_by_percentile = {}

for (
    prediction_percentile,
    prediction_threshold,
) in selected_prediction_thresholds.items():

    signals_pdf = test_scored_pdf.loc[
        test_scored_pdf["prediction"]
        >= prediction_threshold
    ].copy()

    episode_rows_pdf = assign_signal_episodes(
        signals_df=signals_pdf,
        episode_gap_minutes=(
            SIGNAL_EPISODE_GAP_MINUTES
        ),
    )

    if episode_rows_pdf.empty:
        episode_entries_pdf = episode_rows_pdf.copy()
    else:
        episode_entries_pdf = (
            episode_rows_pdf
            .sort_values(
                [
                    GROUP_COLUMN,
                    "kalshi_ticker",
                    "minute_ts",
                ]
            )
            .drop_duplicates(
                subset=["signal_episode_id"],
                keep="first",
            )
            .reset_index(drop=True)
        )

    test_signals_by_percentile[
        prediction_percentile
    ] = signals_pdf

    test_episode_rows_by_percentile[
        prediction_percentile
    ] = episode_rows_pdf

    test_episode_entries_by_percentile[
        prediction_percentile
    ] = episode_entries_pdf


test_signal_count_records = []

for prediction_percentile in (
    SELECTED_OPPORTUNITY_PERCENTILES
):
    signals_pdf = test_signals_by_percentile[
        prediction_percentile
    ]

    episode_entries_pdf = (
        test_episode_entries_by_percentile[
            prediction_percentile
        ]
    )

    test_signal_count_records.append(
        {
            "prediction_percentile": (
                prediction_percentile
            ),
            "validation_derived_threshold": (
                selected_prediction_thresholds[
                    prediction_percentile
                ]
            ),
            "raw_qualifying_row_count": len(
                signals_pdf
            ),
            "signal_episode_count": len(
                episode_entries_pdf
            ),
            "active_game_count": (
                episode_entries_pdf[
                    GROUP_COLUMN
                ].nunique()
                if not episode_entries_pdf.empty
                else 0
            ),
            "active_ticker_count": (
                episode_entries_pdf[
                    "kalshi_ticker"
                ].nunique()
                if not episode_entries_pdf.empty
                else 0
            ),
        }
    )


test_signal_counts_pdf = (
    pd.DataFrame(test_signal_count_records)
    .sort_values(
        by="prediction_percentile"
    )
    .reset_index(drop=True)
)

display(test_signal_counts_pdf)

In [0]:
# Evaluate final test opportunity quality and frequency
# across all selected thresholds

test_total_games = test_scored_pdf[
    GROUP_COLUMN
].nunique()

test_total_tickers = test_scored_pdf[
    "kalshi_ticker"
].nunique()

test_opportunity_summary_records = []

for prediction_percentile in (
    SELECTED_OPPORTUNITY_PERCENTILES
):
    prediction_threshold = (
        selected_prediction_thresholds[
            prediction_percentile
        ]
    )

    signals_pdf = test_signals_by_percentile[
        prediction_percentile
    ]

    episode_entries_pdf = (
        test_episode_entries_by_percentile[
            prediction_percentile
        ]
    )

    raw_signal_count = len(signals_pdf)
    signal_episode_count = len(
        episode_entries_pdf
    )

    if episode_entries_pdf.empty:
        active_game_count = 0
        active_ticker_count = 0

        mean_actual_price_change = np.nan
        median_actual_price_change = np.nan
        positive_return_rate = np.nan
        nonnegative_return_rate = np.nan
        mean_prediction = np.nan
        median_prediction = np.nan

        mean_episodes_per_active_game = np.nan
        median_episodes_per_active_game = np.nan
    else:
        active_game_count = (
            episode_entries_pdf[
                GROUP_COLUMN
            ].nunique()
        )

        active_ticker_count = (
            episode_entries_pdf[
                "kalshi_ticker"
            ].nunique()
        )

        actual_returns = (
            episode_entries_pdf[
                TARGET_COLUMN
            ]
        )

        episode_predictions = (
            episode_entries_pdf[
                "prediction"
            ]
        )

        episodes_by_active_game = (
            episode_entries_pdf
            .groupby(GROUP_COLUMN)[
                "signal_episode_id"
            ]
            .nunique()
        )

        mean_actual_price_change = (
            actual_returns.mean()
        )

        median_actual_price_change = (
            actual_returns.median()
        )

        positive_return_rate = (
            actual_returns.gt(0).mean()
        )

        nonnegative_return_rate = (
            actual_returns.ge(0).mean()
        )

        mean_prediction = (
            episode_predictions.mean()
        )

        median_prediction = (
            episode_predictions.median()
        )

        mean_episodes_per_active_game = (
            episodes_by_active_game.mean()
        )

        median_episodes_per_active_game = (
            episodes_by_active_game.median()
        )

    test_opportunity_summary_records.append(
        {
            "model_name": selected_model_name,
            "prediction_percentile": (
                prediction_percentile
            ),
            "validation_derived_threshold": (
                prediction_threshold
            ),
            "raw_signal_count": (
                raw_signal_count
            ),
            "signal_episode_count": (
                signal_episode_count
            ),
            "total_game_count": (
                test_total_games
            ),
            "active_game_count": (
                active_game_count
            ),
            "game_coverage_rate": (
                active_game_count
                / test_total_games
                if test_total_games > 0
                else np.nan
            ),
            "raw_signals_per_game": (
                raw_signal_count
                / test_total_games
                if test_total_games > 0
                else np.nan
            ),
            "episodes_per_game": (
                signal_episode_count
                / test_total_games
                if test_total_games > 0
                else np.nan
            ),
            "mean_episodes_per_active_game": (
                mean_episodes_per_active_game
            ),
            "median_episodes_per_active_game": (
                median_episodes_per_active_game
            ),
            "total_ticker_count": (
                test_total_tickers
            ),
            "active_ticker_count": (
                active_ticker_count
            ),
            "mean_prediction": (
                mean_prediction
            ),
            "median_prediction": (
                median_prediction
            ),
            "mean_actual_price_change": (
                mean_actual_price_change
            ),
            "median_actual_price_change": (
                median_actual_price_change
            ),
            "positive_return_rate": (
                positive_return_rate
            ),
            "nonnegative_return_rate": (
                nonnegative_return_rate
            ),
        }
    )


test_opportunity_summary_pdf = (
    pd.DataFrame(
        test_opportunity_summary_records
    )
    .sort_values(
        by="prediction_percentile"
    )
    .reset_index(drop=True)
)

display(test_opportunity_summary_pdf)

In [0]:
# Review opportunity counts and outcomes by test game
# for each selected prediction threshold

test_opportunities_by_game_records = []

for prediction_percentile in (
    SELECTED_OPPORTUNITY_PERCENTILES
):
    prediction_threshold = (
        selected_prediction_thresholds[
            prediction_percentile
        ]
    )

    episode_entries_pdf = (
        test_episode_entries_by_percentile[
            prediction_percentile
        ]
    )

    if episode_entries_pdf.empty:
        continue

    percentile_game_summary_pdf = (
        episode_entries_pdf
        .groupby(
            GROUP_COLUMN,
            as_index=False,
        )
        .agg(
            signal_episode_count=(
                "signal_episode_id",
                "nunique",
            ),
            active_ticker_count=(
                "kalshi_ticker",
                "nunique",
            ),
            mean_prediction=(
                "prediction",
                "mean",
            ),
            maximum_prediction=(
                "prediction",
                "max",
            ),
            mean_actual_price_change=(
                TARGET_COLUMN,
                "mean",
            ),
            median_actual_price_change=(
                TARGET_COLUMN,
                "median",
            ),
            positive_return_rate=(
                TARGET_COLUMN,
                lambda series: (
                    series.gt(0).mean()
                ),
            ),
            nonnegative_return_rate=(
                TARGET_COLUMN,
                lambda series: (
                    series.ge(0).mean()
                ),
            ),
        )
    )

    percentile_game_summary_pdf[
        "prediction_percentile"
    ] = prediction_percentile

    percentile_game_summary_pdf[
        "validation_derived_threshold"
    ] = prediction_threshold

    test_opportunities_by_game_records.append(
        percentile_game_summary_pdf
    )


if not test_opportunities_by_game_records:
    test_opportunities_by_game_pdf = (
        pd.DataFrame()
    )

    print(
        "No signal episodes qualified at any "
        "selected threshold on the test split."
    )
else:
    test_opportunities_by_game_pdf = (
        pd.concat(
            test_opportunities_by_game_records,
            ignore_index=True,
        )
        .sort_values(
            by=[
                "prediction_percentile",
                "signal_episode_count",
            ],
            ascending=[
                True,
                False,
            ],
        )
        .reset_index(drop=True)
    )

    display(test_opportunities_by_game_pdf)

In [0]:
# Final supervised modeling summary

dummy_validation_mae = (
    validation_metrics_pdf
    .loc[
        validation_metrics_pdf["model_name"]
        == "dummy_mean",
        "mae",
    ]
    .iloc[0]
)

selected_validation_mae = (
    validation_metrics_pdf
    .loc[
        validation_metrics_pdf["model_name"]
        == selected_model_name,
        "mae",
    ]
    .iloc[0]
)

validation_mae_improvement_rate = (
    (
        dummy_validation_mae
        - selected_validation_mae
    )
    / dummy_validation_mae
    if dummy_validation_mae != 0
    else np.nan
)

print("Supervised modeling summary")
print("===========================")

print(
    f"Selected model: "
    f"{selected_model_name}"
)

print(
    f"Dummy validation MAE: "
    f"{dummy_validation_mae:.6f}"
)

print(
    f"Selected model validation MAE: "
    f"{selected_validation_mae:.6f}"
)

print(
    "Validation MAE improvement over dummy: "
    f"{validation_mae_improvement_rate:.2%}"
)

print(
    f"Test MAE: "
    f"{test_regression_metrics['mae']:.6f}"
)

print(
    f"Test RMSE: "
    f"{test_regression_metrics['rmse']:.6f}"
)

print(
    f"Test R²: "
    f"{test_regression_metrics['r2']:.6f}"
)

print()
print("Validation-derived opportunity tiers")
print("------------------------------------")

for prediction_percentile in (
    SELECTED_OPPORTUNITY_PERCENTILES
):
    threshold = (
        selected_prediction_thresholds[
            prediction_percentile
        ]
    )

    percentile_label = (
        prediction_percentile * 100
    )

    print(
        f"{percentile_label:g}th percentile: "
        f"prediction >= {threshold:.6f}"
    )

print()
print("Final test opportunity results")
print("------------------------------")

display(test_opportunity_summary_pdf)